# Practical 15 — Privacy and Local Models

This notebook runs a **PII de-identification pipeline** end-to-end on a fictional
NovaCorp Financial support ticket: it **detects** personal data with deterministic
rules, **redacts** each identifier to a typed placeholder, and **scores** the
privacy/utility tradeoff against a ground-truth list of seeded PII. Everything is
standard-library and **fully offline** — no network, no API key, no model download,
which is the whole point of the chapter: sensitive text can be processed on a local
machine. This is the Colab-friendly counterpart to the agentic Claude Code / Cline
project, calling the same reference tools in `tools/` directly.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "15-privacy-local-models"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — Load the bundled document and the ground-truth PII

The pipeline operates on `data/customer_support_ticket.txt`, a fictional support
ticket seeded with an email, a US SSN, a phone number, person names, and an
org/account number. The privacy metric will later score the redaction against
`data/seeded_pii.json` — the *ground truth* of exactly which identifiers were
planted — so the score is an honest measure of what the detector caught, not a
number the model graded itself.

In [ ]:
import json
from tools._common import load_doc, load_seeded_pii, seeded_pii_values

document = load_doc()
print(document)

In [ ]:
seeded = load_seeded_pii()      # grouped by type
seeded_values = seeded_pii_values()   # flat list of exact strings
print("Ground-truth PII (seeded), by type:")
print(json.dumps(seeded, indent=2))
print("\nFlattened to", len(seeded_values), "exact identifier strings.")

## Step 1 — Detect personal data (the `detector` agent's tool)

`detect()` applies five explicit rules — regexes for `EMAIL`, `SSN`, `PHONE`, and
`ACCOUNT`, plus a small name gazetteer + titles for `PERSON` — and returns
non-overlapping spans ordered by their position in the text. No model is involved;
detection is deterministic and reproducible.

In [ ]:
from tools.deidentify import detect

entities = detect(document)
print(f"Detected {len(entities)} entities:\n")
for e in entities:
    print(f"  [{e['type']:<7}] {e['value']!r:<35} @ chars {e['start']}-{e['end']}")

## Step 2 — Redact to typed placeholders (the `redactor` agent's tool)

`deidentify()` runs detection and redaction in one call, replacing every detected
span with its typed placeholder (`[EMAIL]`, `[SSN]`, …). The redacted text stays
readable — the surrounding prose is untouched — while each seeded identifier is gone.

In [ ]:
from tools.deidentify import deidentify

result = deidentify(document)
print("Per-type detection counts:")
print(json.dumps(result["counts"], indent=2))
print("\n--- Redacted document ---\n")
print(result["redacted"])

## Step 3 — Score the privacy/utility tradeoff (the `privacy-analyst` agent's tool)

Two reproducible numbers in [0, 1]:

- **privacy** — fraction of the seeded PII that no longer appears in the redacted
  text (1.0 = every known identifier removed). Measured against ground truth, not
  the detector's own output.
- **utility** — fraction of the *non-PII* tokens that survive redaction (1.0 = no
  collateral damage to readable content). Over-redaction is what drags this down.

In [ ]:
from tools.metrics import privacy, utility, tradeoff

priv = privacy(result["redacted"], seeded_values)
util = utility(document, result["redacted"], seeded_values)
scores = tradeoff(document, result["redacted"], seeded_values)

print(f"privacy = {priv:.3f}   (all seeded identifiers removed -> 1.0)")
print(f"utility = {util:.3f}   (non-PII tokens retained)")
print("\ntradeoff (rounded):")
print(json.dumps({"counts": result["counts"], **scores}, indent=2))

## Try it — over-redaction would hurt utility, not help

From the README's "things to try": a benign sentence full of capitalised words
should be left completely alone. A conservative detector keeps `utility` at 1.0;
an aggressive one that nuked every capitalised word would tank it. We confirm the
detector finds **zero** entities here and leaves the text byte-for-byte identical.

In [ ]:
benign = "The Quarterly Review Board met and the Compliance Committee approved the report."
benign_result = deidentify(benign)
print("Entities detected in benign sentence:", benign_result["entities"])
print("Redacted == original?", benign_result["redacted"] == benign)
print("utility on benign text:",
      utility(benign, benign_result["redacted"], seeded_values))

## Try it — add a new identifier and watch the counts change

Another suggestion from the README: append a fresh email and phone number to the
document and re-run the pipeline. The per-type counts rise, and because these new
identifiers are caught too, the redaction stays clean. (We do this in-memory; the
bundled file on disk is unchanged.)

In [ ]:
augmented = document + (
    "\n\nFollow-up: contact backup agent karen.lee@example.com "
    "or call 312-555-0148 for status updates.\n"
)
aug_result = deidentify(augmented)
print("Counts BEFORE:", json.dumps(result["counts"]))
print("Counts AFTER :", json.dumps(aug_result["counts"]))
print("\nNew email/phone present in redacted text?",
      "karen.lee@example.com" in aug_result["redacted"],
      "/", "312-555-0148" in aug_result["redacted"])

## Recap / next steps

We ran the full reference pipeline offline: **load → detect → redact → score**.
On the bundled ticket the detector catches all five PII types, redaction yields
`privacy = 1.0` (every seeded identifier removed) while keeping `utility` high
(the prose survives), and a benign sentence is left untouched so utility is not
wasted on over-redaction.

Next steps to explore on your own:

- Add a document with an **international phone format** and watch privacy expose the
  gap — the US-only regex will miss it.
- Remove a name from the `FIRST_NAMES` gazetteer in `tools/deidentify.py`, re-run,
  and watch `privacy` fall below 1.0 as that identifier leaks through.
- Run the offline test suite with `python -m pytest -q` to see the invariants
  (all types detected, every seeded value removed, utility ≥ 0.9) enforced.